# Option Engine Tutorial (Short-Dated / Weekly)

This notebook shows how to use the option engine for:

1. More accurate single-option pricing (market + smile blend)
2. Combo pricing from leg estimates
3. Stock move -> option move projection (BS re-anchored)
4. Option move target -> required stock move
5. Short-dated risk-free and intraday time-to-expiry tuning

In [4]:
from __future__ import annotations

import sys
from pathlib import Path

from ib_async import IB

# Make project root importable when notebook is opened from `options/`.
repo_root = Path.cwd().resolve()
if repo_root.name == "options":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from options import (
    ComboLegSpec,
    IbAsyncOptionPricingEngine,
    OptionContractSpec,
)

In [ ]:
# --- Connect IB ---
ib = IB()
# Update host/port/clientId as needed for your setup.
ib.connect("127.0.0.1", 4002, clientId=42)
print("Connected:", ib.isConnected())

In [7]:
# Optional: custom short-end risk-free curve for very short-dated options.
def short_end_rate_provider(contract: OptionContractSpec, t_years: float) -> float:
    # Example static anchors (replace with your own live source if wanted).
    # t in years: 1d~0.0027, 1w~0.0192, 2w~0.0385
    if t_years <= 0.01:
        return 0.052
    if t_years <= 0.03:
        return 0.051
    if t_years <= 0.06:
        return 0.050
    return 0.048

engine = IbAsyncOptionPricingEngine(
    ib=ib,
    risk_free_rate=0.05,
    risk_free_rate_provider=short_end_rate_provider,
    expiry_timezone="America/New_York",
    expiry_close_hour=16,
    expiry_close_minute=0,
)
print("Engine ready")

NameError: name 'ib' is not defined

In [8]:
# Define one short-dated contract.
contract = OptionContractSpec(
    symbol="SPY",
    expiry="20260605",  # YYYYMMDD
    strike=540,
    right="C",
)

single_px = engine.estimate_option_price(contract)
print("Estimated option price:", single_px.price)
print("Source:", single_px.source)
print("Confidence:", single_px.confidence)
single_px.details

NameError: name 'engine' is not defined

In [9]:
# Calibrated short-dated risk-free estimate (base curve + market parity blend).
quote = engine.data_provider.get_option_quote(contract)
r_est = engine.estimate_risk_free_rate(contract, quote)
print("Estimated risk-free annualized:", r_est)

# Time-to-expiry in years at second-level precision.
t = engine.time_to_expiry_years(contract, as_of=quote.timestamp)
print("Time to expiry years:", t)
print("Approx hours to expiry:", t * 365 * 24)

NameError: name 'engine' is not defined

In [10]:
# Stock move -> option move (BS re-anchored by default):
# new_option ~= current_market + (BS(S_new)-BS(S_old))
move_up = engine.predict_option_change_for_stock_move(contract, stock_price_change=2.0)
print("Option change for +$2 stock move:", move_up.option_price_change)
print("Estimated new option price:", move_up.estimated_new_option_price)
move_up.details

NameError: name 'engine' is not defined

In [11]:
# Inverse question: how much stock move for +$0.40 option move?
inv = engine.required_stock_move_for_option_change(contract, option_price_change=0.40)
print("Required stock change:", inv.stock_price_change)
print("Projected option new price:", inv.estimated_new_option_price)
inv.details

NameError: name 'engine' is not defined

In [12]:
# Combo example: vertical call spread.
legs = [
    ComboLegSpec(OptionContractSpec("SPY", "20260605", 540, "C"), ratio=1, side="BUY"),
    ComboLegSpec(OptionContractSpec("SPY", "20260605", 545, "C"), ratio=1, side="SELL"),
]
combo_px = engine.estimate_combo_price(legs)
print("Combo estimated price:", combo_px.price)
print("Combo confidence:", combo_px.confidence)

combo_move = engine.predict_combo_change_for_stock_move(legs, stock_price_change=2.0)
print("Combo price change for +$2 move:", combo_move.option_price_change)
print("Combo new estimated price:", combo_move.estimated_new_option_price)

NameError: name 'engine' is not defined